# CLARA Mamba-3 — Kaggle Training Notebook

This notebook trains the 757K-parameter Mamba-3 audio model used by the CLARA plugin suite.

**Model architecture (locked):**
- d_model=128, d_inner=256, nheads=4, headdim=64
- d_state=32, mimo_rank=4, n_layers=6, cond_dim=128
- ~757K params, flat binary export for the custom C++ inference engine

**Expected runtime:**
- P100: ~4-8 hours for 100K steps
- T4x2: similar; single-GPU training is used here

**Before running:**
1. Add the **musdb18hq** dataset from Kaggle's Data Hub. It mounts at `/kaggle/input/musdb18hq/`.
2. Upload `train_clara_mamba3_kaggle.py` to `/kaggle/working/` (Upload button or add as a dataset).
3. Turn on a GPU accelerator (P100 or T4x2) in the notebook settings.

**Note:** This notebook imports the training functions and calls them directly. Do not use `%run train_clara_mamba3_kaggle.py` — the script defaults to on-the-fly audio loading (no precomputed features required).

In [ ]:
# Optional but recommended: soundfile gives faster seeking reads than torchaudio on many files.
!pip install -q soundfile

In [ ]:
import sys
import os

# The adapted training script must be available in /kaggle/working.
sys.path.append('/kaggle/working')

from train_clara_mamba3_kaggle import train, parse_args

args = parse_args([
    '--data', '/kaggle/input/musdb18hq/train',
    '--num-workers', '4',
    '--batch', '4',
    '--grad-accum', '4',
    '--steps', '100000',
])

train(args)

In [ ]:
import os
from train_clara_mamba3_kaggle import export_weights

os.makedirs('/kaggle/working/models', exist_ok=True)
export_weights('/kaggle/working/checkpoints/final.pt',
               '/kaggle/working/models/model_trained.bin')

## Download outputs

Trained artifacts are written to `/kaggle/working/` and available from the **Output** tab:

| File | Size | Purpose |
|------|------|---------|
| `/kaggle/working/checkpoints/final.pt` | ~50 MB | Full training checkpoint |
| `/kaggle/working/checkpoints/best.pt` | ~50 MB | Best checkpoint (early-stop) |
| `/kaggle/working/models/model_trained.bin` | ~3 MB | Flat binary weights for the C++ engine |

Download `model_trained.bin` and copy it into the CLARA `models/` directory, or run `scripts/validate_export.py` to verify the header and dimensions.